# Prior data interactive playground
This notebook explores `prior_data.generate_batch`.


In [9]:
from prior_data import PriorGeneratorConfig, generate_batch
import torch

cfg = PriorGeneratorConfig(seed=18)
batch = generate_batch(cfg, batch_size=10)

for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f'{k}: shape={tuple(v.shape)}, dtype={v.dtype}')
    else:
        print(f'{k}: {v}')


x: shape=(10, 95, 5), dtype=torch.float32
y: shape=(10, 95), dtype=torch.int64
row_mask: shape=(10, 95), dtype=torch.bool
train_test_split_index: shape=(10,), dtype=torch.int64
num_classes: shape=(10,), dtype=torch.int64
unseen_label: 10
removed_class_mask: shape=(10, 10), dtype=torch.bool
seen_class_mask: shape=(10, 10), dtype=torch.bool
removed_class_count: shape=(10,), dtype=torch.int64
removed_class_indices: [tensor([], dtype=torch.int64), tensor([], dtype=torch.int64), tensor([2, 5, 8]), tensor([5]), tensor([], dtype=torch.int64), tensor([1]), tensor([], dtype=torch.int64), tensor([5]), tensor([0]), tensor([2, 3])]


In [10]:
print('train_test_split_index:', batch['train_test_split_index'])
print('num_classes:', batch['num_classes'])
print('row counts:', batch['row_mask'].sum(dim=1))

# Check padding labels
unseen = batch['unseen_label']
y = batch['y']
mask = batch['row_mask']
print('Padding label id:', unseen)
print('Any non-padding rows with unseen label:', ((y==unseen) & mask).any().item())


train_test_split_index: tensor([49, 58, 38, 41, 69, 34, 74, 75, 40, 28])
num_classes: tensor([3, 8, 9, 7, 9, 4, 6, 9, 5, 4])
row counts: tensor([95, 95, 69, 85, 95, 79, 95, 95, 93, 63])
Padding label id: 10
Any non-padding rows with unseen label: False


In [18]:
# Print one task's full X and y (including padding)
b = 3
print('X shape:', batch['x'][b].shape)
print('y shape:', batch['y'][b].shape)
print('X:', batch['x'][b])
print('y:', batch['y'][b])
print(batch['num_classes'][b])

X shape: torch.Size([95, 5])
y shape: torch.Size([95])
X: tensor([[-2.6177e-01,  1.3196e+00,  1.3201e+00,  8.5298e-01, -5.0989e-02],
        [-1.2887e+00, -1.0008e+00, -2.6173e-01,  2.7501e+00,  1.0550e+00],
        [-4.6861e-02,  2.6507e-01,  1.3292e-01,  2.1549e-01,  3.6458e-01],
        [-3.9425e-01,  3.7427e-01, -2.4167e-01, -9.1714e-01, -9.6965e-02],
        [-1.1812e+00, -1.0688e-01, -4.5057e-01,  1.1441e+00,  1.7428e+00],
        [-1.8971e-01,  4.4744e-01, -5.0834e-01,  5.6972e-01,  1.5091e-01],
        [-6.4370e-01,  6.3291e-01, -1.2972e+00, -7.7933e-01,  2.3858e-02],
        [-3.4098e-01, -1.2504e+00, -4.0976e-01,  8.0498e-01, -7.6013e-01],
        [-5.4894e-01, -5.9128e-01, -8.9711e-02, -2.7042e-01,  3.2261e-02],
        [-7.3576e-01,  5.2317e-01, -1.8043e+00, -1.7033e+00,  9.8477e-01],
        [-1.0083e+00, -1.2738e-01,  1.0165e+00,  1.0035e+00, -7.6750e-01],
        [ 1.5540e+00,  1.7689e-01, -1.3756e+00, -2.4031e-01,  3.7423e-01],
        [-1.4772e+00,  1.1401e+00, -1.6616

In [12]:
# Inspect padded entries (rows where row_mask is False)
b = 0
mask_b = batch['row_mask'][b]
pad_idx = (~mask_b).nonzero(as_tuple=True)[0]
print('task', b, 'pad rows:', pad_idx.tolist())
if pad_idx.numel() > 0:
    y_pad = batch['y'][b][pad_idx]
    x_pad = batch['x'][b][pad_idx]
    print('y padded unique:', torch.unique(y_pad).tolist())
    print('x padded row sample (first):', x_pad[0])
else:
    print('no padding for this task')

# Verify all padding rows use unseen_label
unseen = batch['unseen_label']
print('all padding labels == unseen_label:', (batch['y'][b][~mask_b] == unseen).all().item())


task 0 pad rows: []
no padding for this task
all padding labels == unseen_label: True


In [13]:
# Show removed classes per task
for b in range(batch['x'].shape[0]):
    print(f'task {b}: removed_count={batch["removed_class_count"][b].item()}, ', end='')
    print('removed_classes=', batch['removed_class_indices'][b].tolist())


task 0: removed_count=0, removed_classes= []
task 1: removed_count=0, removed_classes= []
task 2: removed_count=3, removed_classes= [2, 5, 8]
task 3: removed_count=1, removed_classes= [5]
task 4: removed_count=0, removed_classes= []
task 5: removed_count=1, removed_classes= [1]
task 6: removed_count=0, removed_classes= []
task 7: removed_count=1, removed_classes= [5]
task 8: removed_count=1, removed_classes= [0]
task 9: removed_count=2, removed_classes= [2, 3]


In [14]:
# Sanity check: removed classes <= half of K
for b in range(batch['x'].shape[0]):
    K = batch['num_classes'][b].item()
    max_removed = min(K - 1, K // 2)
    removed = batch['removed_class_count'][b].item()
    print(f'task {b}: K={K}, removed={removed}, max_removed={max_removed}')


task 0: K=3, removed=0, max_removed=1
task 1: K=8, removed=0, max_removed=4
task 2: K=9, removed=3, max_removed=4
task 3: K=7, removed=1, max_removed=3
task 4: K=9, removed=0, max_removed=4
task 5: K=4, removed=1, max_removed=2
task 6: K=6, removed=0, max_removed=3
task 7: K=9, removed=1, max_removed=4
task 8: K=5, removed=1, max_removed=2
task 9: K=4, removed=2, max_removed=2


In [15]:
# Visualize one task's train/test split and removed/seen classes
b = 1
split = batch['train_test_split_index'][b].item()
y_b = batch['y'][b] 
mask_b = batch['row_mask'][b]
print('split:', split)
print('train labels (unique):', torch.unique(y_b[:split][mask_b[:split]]))
print('test labels (unique):', torch.unique(y_b[split:][mask_b[split:]]))
print('removed classes:', torch.where(batch['removed_class_mask'][b])[0].tolist())
print('seen classes:', torch.where(batch['seen_class_mask'][b])[0].tolist())


split: 58
train labels (unique): tensor([0, 1, 3, 4, 6, 7])
test labels (unique): tensor([0, 1, 2, 3, 4, 6, 7])
removed classes: []
seen classes: [0, 1, 3, 4, 6, 7]
